In [1]:
import time
import os
import base64
import csv
import math
from zk_normal import Election, Ballot, readCsvData, FILE_PATH
from zk_no_merkle import ZKVotingSystem
from zk_merkle import ZKMerkleVotingSystem, ZKMerkleTree, calculate_optimal_depth


In [2]:
import psutil

def get_memory_mb():
    """獲取當前 Python 行程佔用的實體記憶體 (MB)"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

In [3]:


def mock_decrypt(enc_vote):
    return int(base64.b64decode(enc_vote.encode('utf-8')).decode('utf-8'))
  

In [4]:
def run_benchmark(test_amount, csv_writer, csv_file):
    print(f"\n{'='*50}")
    print(f"=== Benchmark: N = {test_amount} Voters ===")
    print(f"{'='*50}")

    voters = readCsvData(FILE_PATH['csv'])
    candidates = [1, 2]

    voter_data = []

    # Assume voters array has enough data as guaranteed by user
    for i in range(test_amount):
        voter_data.append({
            "user": voters[i],
            "secret": os.urandom(31).hex(),
            "choice": candidates[i % len(candidates)],
            "index": i
        })

    print(
        f"Total voters ready: {len(voter_data)}. Starting full ZKP generation...")

    # ---------------------------------------------------------
    # 1. Baseline: No-Merkle System
    # ---------------------------------------------------------
    print("\n[Running] Baseline: No-Merkle System...")

    # Measure initial memory
    mem_before_nm = get_memory_mb()

    sys_no_merkle = ZKVotingSystem(vkey_path=FILE_PATH['vkey_no_merkle'])
    election_no_merkle = Election(
        name=f"No_Merkle_{test_amount}", candidates=candidates)

    for data in voter_data:
        comm = sys_no_merkle.computeIdentityCommitment(
            data["user"], data["secret"])
        sys_no_merkle.registerVoterCommitment(
            election_no_merkle, data["user"].hashId, comm)

    # Measure memory after registration
    mem_after_nm = get_memory_mb()
    nm_memory_used = max(0, mem_after_nm - mem_before_nm)

    nm_prove_time = 0
    nm_verify_time = 0

    for idx, data in enumerate(voter_data):
        voter = data["user"]
        enc_vote = base64.b64encode(
            str(data["choice"]).encode('utf-8')).decode('utf-8')

        start = time.perf_counter()
        zk_data = sys_no_merkle.generateVoteProof(voter, data["secret"])
        nm_prove_time += (time.perf_counter() - start)

        ballot = Ballot(enc_vote, zk_data["proof"], zk_data["public_signals"])

        start = time.perf_counter()
        sys_no_merkle.castVote(election_no_merkle, voter, ballot)
        nm_verify_time += (time.perf_counter() - start)

        # Print progress every 50 iterations to avoid thinking the terminal is frozen
        if (idx + 1) % 50 == 0 or (idx + 1) == test_amount:
            print(f"  -> No-Merkle Progress: {idx + 1} / {test_amount}")

    avg_nm_prove = nm_prove_time / test_amount
    avg_nm_verify = nm_verify_time / test_amount

    # ---------------------------------------------------------
    # 2. Proposed: Merkle Tree System
    # ---------------------------------------------------------
    print("\n[Running] Proposed: Merkle Tree System...")

    depth = calculate_optimal_depth(test_amount)

    # Measure initial memory
    mem_before_m = get_memory_mb()

    sys_merkle = ZKMerkleVotingSystem()
    election_merkle = Election(
        name=f"Merkle_{test_amount}", candidates=candidates)

    registered_commitments = []
    for data in voter_data:
        comm = sys_merkle.computeIdentityCommitment(
            data["user"], data["secret"])
        registered_commitments.append(comm)
        sys_merkle.registerVoterStatus(election_merkle, data["user"].hashId)

    tree = ZKMerkleTree(registered_commitments, depth)
    global_root = tree.get_root()

    # Measure memory after Merkle Tree construction
    mem_after_m = get_memory_mb()
    m_memory_used = max(0, mem_after_m - mem_before_m)
    print(f"Merkle Tree Depth: {depth}, RAM used: {m_memory_used:.2f} MB")

    m_prove_time = 0
    m_verify_time = 0

    for idx, data in enumerate(voter_data):
        voter = data["user"]
        enc_vote = base64.b64encode(
            str(data["choice"]).encode('utf-8')).decode('utf-8')

        path = tree.get_path(data["index"])

        start = time.perf_counter()
        # Ensure generateVoteProof handles the depth parameter for dynamic circuit loading
        zk_data = sys_merkle.generateVoteProof(
            voter, data["secret"], path, global_root, depth)
        m_prove_time += (time.perf_counter() - start)

        ballot = Ballot(enc_vote, zk_data["proof"], zk_data["public_signals"])

        start = time.perf_counter()
        sys_merkle.castVote(election_merkle, voter, ballot, depth)
        m_verify_time += (time.perf_counter() - start)

        if (idx + 1) % 50 == 0 or (idx + 1) == test_amount:
            print(f"  -> Merkle Tree Progress: {idx + 1} / {test_amount}")

    avg_m_prove = m_prove_time / test_amount
    avg_m_verify = m_verify_time / test_amount

    print(f"\n[Results for N={test_amount}]")
    print(
        f"No-Merkle -> Avg Prove: {avg_nm_prove:.4f}s, Avg Verify: {avg_nm_verify:.4f}s, RAM: {nm_memory_used:.2f}MB")
    print(
        f"Merkle    -> Avg Prove: {avg_m_prove:.4f}s, Avg Verify: {avg_m_verify:.4f}s, RAM: {m_memory_used:.2f}MB")

    # Write data to CSV and flush immediately
    csv_writer.writerow({
        "N_Voters": test_amount,
        "Depth": depth,
        "NM_Avg_Prove_s": f"{avg_nm_prove:.4f}",
        "NM_Avg_Verify_s": f"{avg_nm_verify:.4f}",
        "NM_RAM_MB": f"{nm_memory_used:.2f}",
        "M_Avg_Prove_s": f"{avg_m_prove:.4f}",
        "M_Avg_Verify_s": f"{avg_m_verify:.4f}",
        "M_RAM_MB": f"{m_memory_used:.2f}"
    })
    csv_file.flush()
    os.fsync(csv_file.fileno())

In [ ]:
if __name__ == "__main__":
    test_amount = [128, 256, 512]

    csv_filename = "zkp_benchmark_results.csv"
    file_exists = os.path.isfile(csv_filename)

    with open(csv_filename, 'a', newline='') as csvfile:
        fieldnames = [
            "N_Voters", "Depth",
            "NM_Avg_Prove_s", "NM_Avg_Verify_s", "NM_RAM_MB",
            "M_Avg_Prove_s", "M_Avg_Verify_s", "M_RAM_MB"
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        if not file_exists:
            writer.writeheader()

        for i in test_amount:
            run_benchmark(i, writer, csvfile)

    print("\nBenchmark completed. All results saved to zkp_benchmark_results.csv")


=== Benchmark: N = 256 Voters ===
Total voters ready: 256. Starting full ZKP generation...

[Running] Baseline: No-Merkle System...
  -> No-Merkle Progress: 50 / 256
  -> No-Merkle Progress: 100 / 256
[SnarkJS Error] Proof not eligible: 


ValueError: ZK_VERIFICATION_FAILED